In [ ]:
import os
from pathlib import Path
from collections import Counter

def check_raw_folders(data_folder):
    """
    Check files inside folders starting with 'Raw'
    """
    print(f"CHECKING FOLDERS STARTING WITH 'Raw' IN: {data_folder}")
    
    if not os.path.exists(data_folder):
        print(f" Folder does not exist: {data_folder}")
        return None
    
    # Get all folders starting with 'Raw'
    all_items = os.listdir(data_folder)
    raw_folders = sorted([item for item in all_items 
                         if os.path.isdir(os.path.join(data_folder, item)) 
                         and item.startswith('Raw')])
    
    print(f"\n Found {len(raw_folders)} folders starting with 'Raw'\n")
    
    # Store all file information
    all_files_info = []
    
    # Check each Raw folder
    for folder in raw_folders:
        folder_path = os.path.join(data_folder, folder)
        
        # Get all files in this folder
        files = [f for f in os.listdir(folder_path) 
                if os.path.isfile(os.path.join(folder_path, f))]
        
        print(f" {folder}")
        print(f"   Total files: {len(files)}")
        
        # Group by file extension
        extensions = Counter([os.path.splitext(f)[1].lower() for f in files])
        for ext, count in extensions.items():
            ext_display = ext if ext else "(no extension)"
            print(f"   {ext_display}: {count} files")
        
        # Show first 5 files as sample
        if files:
            print(f"   Sample files:")
            for f in files[:5]:
                print(f"      - {f}")
            if len(files) > 5:
                print(f"      ... and {len(files) - 5} more")
        
        print()
        
        # Store info
        for f in files:
            file_path = os.path.join(folder_path, f)
            all_files_info.append({
                'folder': folder,
                'filename': f,
                'extension': os.path.splitext(f)[1].lower(),
                'size_kb': os.path.getsize(file_path) / 1024,
                'path': file_path
            })
    
    print(f"Total Raw folders: {len(raw_folders)}")
    print(f"Total files in Raw folders: {len(all_files_info)}")
    
    # Count by extension
    all_extensions = Counter([info['extension'] for info in all_files_info])
    print(f"\nFiles by type:")
    for ext, count in all_extensions.most_common():
        ext_display = ext if ext else "(no extension)"
        print(f"  {ext_display}: {count} files")
    
    return all_files_info, raw_folders


def get_file_list_by_folder(data_folder):
    """
    Get a dictionary of {folder_name: [list of files]}
    """
    
    result = {}
    
    all_items = os.listdir(data_folder)
    raw_folders = [item for item in all_items 
                   if os.path.isdir(os.path.join(data_folder, item)) 
                   and item.startswith('Raw')]
    
    for folder in sorted(raw_folders):
        folder_path = os.path.join(data_folder, folder)
        files = [f for f in os.listdir(folder_path) 
                if os.path.isfile(os.path.join(folder_path, f))]
        result[folder] = sorted(files)
    
    return result


def save_file_list(data_folder, output_file="raw_folders_files.txt"):
    """
    Save the file list to a text file
    """
    
    file_dict = get_file_list_by_folder(data_folder)
    
    with open(output_file, 'w') as f:
        for folder, files in file_dict.items():
            f.write(f"\n{'='*60}\n")
            f.write(f"{folder} ({len(files)} files)\n")
            f.write(f"{'='*60}\n")
            for file in files:
                f.write(f"  {file}\n")
    
    print(f"\n File list saved to: {output_file}")


if __name__ == "__main__":
    DATA_FOLDER = "./data"
    # Check all Raw folders
    all_files_info, raw_folders = check_raw_folders(DATA_FOLDER)
    # Save to file
    if all_files_info:
        save_file_list(DATA_FOLDER)
    # Get dictionary of files by folder
    file_dict = get_file_list_by_folder(DATA_FOLDER)
    
    print(f"\n You can access files by folder using: file_dict['folder_name']")

CHECKING FOLDERS STARTING WITH 'Raw' IN: ./data

✓ Found 19 folders starting with 'Raw'

 Raw_Recordings_Day1_pt11
   Total files: 13
   .wav: 13 files
   Sample files:
      - 20211120_225737_192.wav
      - 20211120_230237_192.wav
      - 20211120_230738_192.wav
      - 20211120_231238_192.wav
      - 20211120_231739_192.wav
      ... and 8 more

 Raw_Recordings_Day2_pt1
   Total files: 15
   .wav: 15 files
   Sample files:
      - 20211121_000245_192.wav
      - 20211121_000746_192.wav
      - 20211121_001246_192.wav
      - 20211121_001747_192.wav
      - 20211121_002248_192.wav
      ... and 10 more

 Raw_Recordings_Day2_pt2
   Total files: 15
   .wav: 15 files
   Sample files:
      - 20211121_011754_192.wav
      - 20211121_012254_192.wav
      - 20211121_012755_192.wav
      - 20211121_013255_192.wav
      - 20211121_013756_192.wav
      ... and 10 more

 Raw_Recordings_Day2_pt3
   Total files: 15
   .wav: 15 files
   Sample files:
      - 20211121_023302_192.wav
      - 202111

Comparing labels

In [ ]:
import os
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import re

def parse_filename_timestamp(filename):
    """
    Extract timestamp from filename like: 20211120_102109_192
    Format: YYYYMMDD_HHMMSS_XXX
    Returns datetime object (ignoring seconds for comparison)
    """
    try:

        name = os.path.splitext(filename)[0]
        # Extract date and time parts (YYYYMMDD_HHMMSS)
        match = re.match(r'(\d{8})_(\d{6})_.*', name)
        if match:
            date_part = match.group(1)  # YYYYMMDD
            time_part = match.group(2)  # HHMMSS
            
            # Parse to datetime
            dt = datetime.strptime(f"{date_part}{time_part}", "%Y%m%d%H%M%S")
            return dt
        return None
    except Exception as e:
        print(f"    Error parsing {filename}: {e}")
        return None


def parse_excel_timestamp(timestamp_str):
    """
    Parse various Excel timestamp formats:
    - "11/20/2021 10:21:09 AM"
    - "2021-11-20 19:01:09 UTC"
    - Other common formats
    Returns datetime object
    """
    if pd.isna(timestamp_str):
        return None
    
    # If already datetime
    if isinstance(timestamp_str, datetime):
        return timestamp_str
    
    timestamp_str = str(timestamp_str).strip()
    
    # Try different formats
    formats = [
        "%m/%d/%Y %I:%M:%S %p",      
        "%Y-%m-%d %H:%M:%S UTC",     
        "%Y-%m-%d %H:%M:%S",         
        "%m/%d/%Y %H:%M:%S",          
        "%Y/%m/%d %H:%M:%S",          
        "%d/%m/%Y %H:%M:%S",         
    ]
    
    for fmt in formats:
        try:
            return datetime.strptime(timestamp_str, fmt)
        except ValueError:
            continue
    
    print(f"    Could not parse timestamp: {timestamp_str}")
    return None


def timestamps_match(dt1, dt2, minute_tolerance=1):
    """
    Check if two timestamps match within tolerance
    Ignores seconds, allows minute difference up to minute_tolerance
    """
    if dt1 is None or dt2 is None:
        return False
    
    # Check if year, month, day match
    if dt1.year != dt2.year or dt1.month != dt2.month or dt1.day != dt2.day:
        return False
    
    # Check if hour matches
    if dt1.hour != dt2.hour:
        return False
    
    # Check minute difference
    minute_diff = abs(dt1.minute - dt2.minute)
    
    return minute_diff <= minute_tolerance


def match_files_with_labels(data_folder, excel_file, timestamp_column, minute_tolerance=1):
    """
    Match raw folder files with Excel labels
    
    Parameters:
    - data_folder: Path to folder containing Raw_* folders
    - excel_file: Path to Excel file with labels
    - timestamp_column: Name of column containing timestamps in Excel
    - minute_tolerance: Allowed minute difference (default: 1)
    """
    
    print("MATCHING RAW FILES WITH EXCEL LABELS")

    
    # Read Excel file
    print(f"\n Reading Excel file: {excel_file}")
    df_labels = pd.read_excel(excel_file)
    print(f"   Loaded {len(df_labels)} rows")
    print(f"   Columns: {list(df_labels.columns)}")
    
    # Check if timestamp column exists
    if timestamp_column not in df_labels.columns:
        print(f"\n Error: Column '{timestamp_column}' not found in Excel file!")
        print(f"   Available columns: {list(df_labels.columns)}")
        return None
    
    # Parse Excel timestamps
    print(f"\n Parsing timestamps from column: '{timestamp_column}'")
    df_labels['parsed_timestamp'] = df_labels[timestamp_column].apply(parse_excel_timestamp)
    valid_timestamps = df_labels['parsed_timestamp'].notna().sum()
    print(f"   Successfully parsed {valid_timestamps}/{len(df_labels)} timestamps")
    
    # Get all Raw folders
    all_items = os.listdir(data_folder)
    raw_folders = sorted([item for item in all_items 
                         if os.path.isdir(os.path.join(data_folder, item)) 
                         and item.startswith('Raw')])
    
    print(f"\n Found {len(raw_folders)} Raw folders")
    
    # Collect all files from Raw folders
    results = []
    matched_count = 0
    unknown_count = 0
    
    for folder in raw_folders:
        folder_path = os.path.join(data_folder, folder)
        files = [f for f in os.listdir(folder_path) 
                if os.path.isfile(os.path.join(folder_path, f))]
        
        print(f"\n Processing: {folder} ({len(files)} files)")
        
        for filename in files:
            file_timestamp = parse_filename_timestamp(filename)
            
            if file_timestamp is None:
                # Could not parse filename
                results.append({
                    'folder': folder,
                    'filename': filename,
                    'status': 'unknown',
                    'matched_label': None,
                    'file_timestamp': None,
                    'label_timestamp': None,
                    'reason': 'Could not parse filename timestamp'
                })
                unknown_count += 1
                continue
            
            # Try to find match in Excel
            match_found = False
            best_match = None
            
            for idx, row in df_labels.iterrows():
                label_timestamp = row['parsed_timestamp']
                
                if timestamps_match(file_timestamp, label_timestamp, minute_tolerance):
                    match_found = True
                    best_match = row
                    break
            
            if match_found:
                # Create result dictionary with all Excel columns
                result = {
                    'folder': folder,
                    'filename': filename,
                    'status': 'matched',
                    'file_timestamp': file_timestamp.strftime('%Y-%m-%d %H:%M:%S'),
                    'label_timestamp': best_match['parsed_timestamp'].strftime('%Y-%m-%d %H:%M:%S'),
                }
                # Add all columns from Excel
                for col in df_labels.columns:
                    if col != 'parsed_timestamp':
                        result[f'label_{col}'] = best_match[col]
                
                results.append(result)
                matched_count += 1
            else:
                results.append({
                    'folder': folder,
                    'filename': filename,
                    'status': 'unknown',
                    'matched_label': None,
                    'file_timestamp': file_timestamp.strftime('%Y-%m-%d %H:%M:%S'),
                    'label_timestamp': None,
                    'reason': 'No matching timestamp in Excel'
                })
                unknown_count += 1
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    
    # Print summary
    print("MATCHING SUMMARY")
    print(f"Total files processed: {len(results)}")
    print(f" Matched: {matched_count} ({matched_count/len(results)*100:.1f}%)")
    print(f" Unknown: {unknown_count} ({unknown_count/len(results)*100:.1f}%)")
    
    return results_df


def save_results(results_df, output_file="matched_files.csv"):
    """
    Save results to CSV
    """
    if results_df is not None:
        results_df.to_csv(output_file, index=False)
        print(f"\n Results saved to: {output_file}")
        
        # Show preview
        print(f"\n Preview of results:")
        print(results_df.head(10).to_string())
        
        return True
    return False

 # CONFIGURATION
if __name__ == "__main__":
    # Path to your data folder containing Raw_* folders
    DATA_FOLDER = "./data"
    # Path to your Excel file with labels
    EXCEL_FILE = "./122870.xlsx"  # Change this
    # Name of the column in Excel that contains timestamps
    TIMESTAMP_COLUMN = "interval_start"  # Change this to your actual column name
    # Minute tolerance for matching (default: 1 minute)
    MINUTE_TOLERANCE = 1
    # Output CSV file name
    OUTPUT_FILE = "matched_files.csv"
    # Run matching
    results_df = match_files_with_labels(
        data_folder=DATA_FOLDER,
        excel_file=EXCEL_FILE,
        timestamp_column=TIMESTAMP_COLUMN,
        minute_tolerance=MINUTE_TOLERANCE
    )
    
    # Save results
    if results_df is not None:
        save_results(results_df, OUTPUT_FILE)
        
        # Show statistics by status
        print("\n Status breakdown:")
        print(results_df['status'].value_counts())
        
        # Show unknown files
        unknown_files = results_df[results_df['status'] == 'unknown']
        if len(unknown_files) > 0:
            print(f"\n  Unknown files (first 10):")
            print(unknown_files[['folder', 'filename', 'reason']].head(10).to_string())

MATCHING RAW FILES WITH EXCEL LABELS

 Reading Excel file: ./122870.xlsx
   Loaded 176 rows
   Columns: ['interval_start', 'total_count', 'CT_count', 'FB_count', 'PBS_count', 'activity', 'act_type']

 Parsing timestamps from column: 'interval_start'
   Successfully parsed 176/176 timestamps

 Found 19 Raw folders

 Processing: Raw_Recordings_Day1_pt11 (13 files)

 Processing: Raw_Recordings_Day2_pt1 (15 files)

 Processing: Raw_Recordings_Day2_pt2 (15 files)

 Processing: Raw_Recordings_Day2_pt3 (15 files)

 Processing: Raw_Recordings_Day2_pt4 (15 files)

 Processing: Raw_Recordings_Day2_pt5 (15 files)

 Processing: Raw_Recordings_Day2_pt6 (15 files)

 Processing: Raw_Recordings_Day2_pt7 (15 files)

 Processing: Raw_Recordings_Day2_pt8 (18 files)

 Processing: Raw_recordings_Day1_pt1 (16 files)

 Processing: Raw_recordings_Day1_pt10 (15 files)

 Processing: Raw_recordings_Day1_pt2 (15 files)

 Processing: Raw_recordings_Day1_pt3 (15 files)

 Processing: Raw_recordings_Day1_pt4 (15 file

In [3]:
import pandas as pd
from datetime import datetime

def fix_matched_csv(input_csv, output_csv):
    """
    Fix the matched CSV file:
    1. Fill EMPTY/NaN cells with 'unknown' for unmatched rows
    2. Sort by timestamp in chronological order
    """
    
    # Read CSV
    print(f"\n Reading: {input_csv}")
    df = pd.read_csv(input_csv)
    print(f"   Total rows: {len(df)}")
    
    # Show current status distribution
    print(f"\n Current status distribution:")
    print(df['status'].value_counts())
    
    # Fill EMPTY cells with 'unknown' for unmatched rows
    print(f"\n Filling empty cells with 'unknown' in unmatched rows...")
    
    # For rows where status is 'unknown', fill empty/NaN cells with 'unknown'
    unknown_mask = df['status'] == 'unknown'
    
    # Get all columns except the ones we want to keep their original values
    columns_to_check = [col for col in df.columns 
                        if col not in ['folder', 'filename', 'status', 'file_timestamp']]
    
    # Count empty cells before filling
    empty_count_before = 0
    for col in columns_to_check:
        empty_count_before += (unknown_mask & df[col].isna()).sum()
    
    print(f"   Found {empty_count_before} empty cells in unknown rows")
    
    # Fill only empty/NaN values with 'unknown' for these columns in unknown rows
    for col in columns_to_check:
        df.loc[unknown_mask & df[col].isna(), col] = 'unknown'
    
    # Count filled cells
    filled_count = 0
    for col in columns_to_check:
        filled_count += (unknown_mask & (df[col] == 'unknown')).sum()
    
    print(f"    Filled {empty_count_before} empty cells with 'unknown'")
    
    # Convert file_timestamp to datetime for proper sorting
    print(f"\n Sorting by timestamp...")
    
    # Parse file_timestamp to datetime
    df['timestamp_sort'] = pd.to_datetime(df['file_timestamp'], format='%m/%d/%Y %H:%M', errors='coerce')
    
    # Sort by timestamp in chronological order
    df_sorted = df.sort_values('timestamp_sort', ascending=True)
    
    # Drop the temporary sort column
    df_sorted = df_sorted.drop('timestamp_sort', axis=1)
    
    # Reset index
    df_sorted = df_sorted.reset_index(drop=True)
    
    # Save fixed CSV
    print(f"\n Saving to: {output_csv}")
    df_sorted.to_csv(output_csv, index=False)
    print(f"Total rows: {len(df_sorted)}")
    print(f"Matched: {(df_sorted['status'] == 'matched').sum()}")
    print(f"Unknown: {(df_sorted['status'] == 'unknown').sum()}")
    
    # Show timestamp range
    first_timestamp = df_sorted['file_timestamp'].iloc[0]
    last_timestamp = df_sorted['file_timestamp'].iloc[-1]
    print(f"\nTimestamp range:")
    print(f"  First: {first_timestamp}")
    print(f"  Last: {last_timestamp}")
    
    # Show first few rows
    print(f"\n First 5 rows (in time order):")
    print(df_sorted[['filename', 'file_timestamp', 'status']].head(5).to_string(index=False))
    
    # Show sample UNKNOWN rows
    print(f"\n Sample UNKNOWN rows (showing all columns):")
    unknown_sample = df_sorted[df_sorted['status'] == 'unknown'].head(2)
    if len(unknown_sample) > 0:
        for idx, row in unknown_sample.iterrows():
            print(f"\n  Row {idx}:")
            for col in df_sorted.columns:
                value = row[col] if pd.notna(row[col]) else 'NaN'
                print(f"    {col}: {value}")
    
    # Verify no NaN values remain in unknown rows
    print(f"\n Verification:")
    unknown_rows = df_sorted[df_sorted['status'] == 'unknown']
    
    if len(unknown_rows) > 0:
        nan_found = False
        for col in columns_to_check:
            nan_count = unknown_rows[col].isna().sum()
            if nan_count > 0:
                print(f"     Column '{col}' still has {nan_count} NaN values")
                nan_found = True
        
        if not nan_found:
            print(f"    All empty cells in unknown rows are filled with 'unknown'")
    
    return df_sorted


if __name__ == "__main__":
    INPUT_CSV = "matched_files.csv"
    OUTPUT_CSV = "matched_files_fixed.csv"
    
    # Run the fix
    df_fixed = fix_matched_csv(INPUT_CSV, OUTPUT_CSV)


 Reading: matched_files.csv
   Total rows: 287

 Current status distribution:
status
unknown    145
matched    142
Name: count, dtype: int64

 Filling empty cells with 'unknown' in unmatched rows...
   Found 1305 empty cells in unknown rows
    Filled 1305 empty cells with 'unknown'

 Sorting by timestamp...

 Saving to: matched_files_fixed.csv
Total rows: 287
Matched: 142
Unknown: 145

Timestamp range:
  First: 2021-11-20 22:57:37
  Last: 2021-11-20 21:37:27

 First 5 rows (in time order):
               filename      file_timestamp  status
20211120_225737_192.wav 2021-11-20 22:57:37 matched
20211120_230237_192.wav 2021-11-20 23:02:37 matched
20211120_230738_192.wav 2021-11-20 23:07:38 matched
20211120_231238_192.wav 2021-11-20 23:12:38 matched
20211120_231739_192.wav 2021-11-20 23:17:39 matched

 Sample UNKNOWN rows (showing all columns):

  Row 39:
    folder: Raw_Recordings_Day2_pt2
    filename: 20211121_021300_192.wav
    status: unknown
    file_timestamp: 2021-11-21 02:13:00
 

C:\Users\ukart\AppData\Local\Temp\ipykernel_5852\2973288052.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unknown' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[unknown_mask & df[col].isna(), col] = 'unknown'
C:\Users\ukart\AppData\Local\Temp\ipykernel_5852\2973288052.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unknown' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[unknown_mask & df[col].isna(), col] = 'unknown'
C:\Users\ukart\AppData\Local\Temp\ipykernel_5852\2973288052.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unknown' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df